# 02 · 文献计量分析

基于 `01_parse_nbib.ipynb` 清洗好的数据做分析，并产出 VOSviewer 的输入文件。

- 输入：`data/interim/articles.csv`（一篇文章一行）、`data/interim/authors.csv`（一篇 × 一个作者）
- 分析口径（**待定**）：年份范围 —— 现在 `year` 基于 DEP，覆盖 2018–2026


## 第一步：读取清洗好的数据

确认两张表能正常读入、字段齐全，后面每一步都在这个基础上做。

> 用 `encoding='utf-8-sig'`（保存时带了 BOM）；`keep_default_na=False` 让缺失值保持空字符串，方便按字符串处理。


In [1]:
import os
import pandas as pd

interim_dir = os.path.join(os.getcwd(), '..', 'data', 'interim')

articles = pd.read_csv(os.path.join(interim_dir, 'articles.csv'),
                       encoding='utf-8-sig', keep_default_na=False)
authors = pd.read_csv(os.path.join(interim_dir, 'authors.csv'),
                      encoding='utf-8-sig', keep_default_na=False)

print('articles:', articles.shape)
print('authors :', authors.shape)

print('\narticles 列:', list(articles.columns))
print('\nauthors 列 :', list(authors.columns))

articles[['pmid', 'year', 'first_author', 'n_authors', 'journal', 'doi']].head()


articles: (29907, 22)
authors : (275392, 6)

articles 列: ['pmid', 'doi', 'title', 'abstract', 'authors', 'first_author', 'last_author', 'n_authors', 'orcids', 'affiliations', 'mesh', 'keywords', 'pubtypes', 'journal', 'pubdate', 'year', 'language', 'volume', 'issue', 'pages', 'source_file', 'authors_key']

authors 列 : ['pmid', 'seq', 'full_name', 'orcid', 'affiliations', 'author_key']


,pmid,year,first_author,n_authors,journal,doi
0,33823184,2021,"Sun, Huiling",5,European journal of pharmacology,10.1016/j.ejphar.2021.174072
1,33812273,2021,"Kou, Xing",3,Ultrasonics,10.1016/j.ultras.2021.106426
2,33762094,2021,"Feng, Yong",6,ISA transactions,10.1016/j.isatra.2021.03.013
3,33760976,2021,"Zhao, Jia-Min",9,International journal of legal medicine,10.1007/s00414-021-02559-2
4,33744670,2021,"Cao, Shuai",6,Biosensors & bioelectronics,10.1016/j.bios.2021.113164


## 第二步：作者关键词子集（只保留有 `keywords` 的文章）

按你的要求，先做**只用作者关键词（OT）**的子集：去掉没有作者关键词的论文。

> 这里只做过滤 + 看规模，**还没做任何关键词清洗**（大小写、单复数、MeSH 限定词等口径待定）。


In [2]:
kw_articles = articles[articles['keywords'] != ''].copy()

print('全部文章    :', len(articles))
print('有作者关键词:', len(kw_articles), f'({len(kw_articles) / len(articles) * 100:.1f}%)')
print('被去掉      :', len(articles) - len(kw_articles))

tokens = kw_articles['keywords'].apply(lambda s: s.split('||')).explode()
print()
print('关键词出现次数（含重复）:', len(tokens))
print('不同写法（未清洗）      :', tokens.nunique())

print()
print('示例（前 3 篇）：')
for kw in kw_articles['keywords'].head(3):
    print('  -', kw.split('||'))


全部文章    : 29907
有作者关键词: 23480 (78.5%)
被去掉      : 6427

关键词出现次数（含重复）: 116671
不同写法（未清洗）      : 65171

示例（前 3 篇）：
  - ['Exendin ((9-39))', 'Food intake', 'Neurotransmitter', 'Nucleus tractus solitarius (NTS)', 'SHU 9119']
  - ['Closed surface crack', 'Laser ultrasonics', 'Nonlinear ultrasonic testing method', 'Surface acoustic wave']
  - ['Attention', 'Fault diagnosis', 'Few-shot classification', 'Meta-learning', 'Semi-supervised learning']


## 第三步：MeSH 子集（只保留有 `mesh` 的文章）

同理，做**只用 MeSH 主题词**的子集：去掉没有 MeSH 的论文。


In [3]:
mesh_articles = articles[articles['mesh'] != ''].copy()

print('全部文章:', len(articles))
print('有 MeSH :', len(mesh_articles), f'({len(mesh_articles) / len(articles) * 100:.1f}%)')
print('被去掉  :', len(articles) - len(mesh_articles))

tokens = mesh_articles['mesh'].apply(lambda s: s.split('||')).explode()
print()
print('主题词出现次数（含重复）:', len(tokens))
print('不同写法（未清洗）      :', tokens.nunique())

print()
print('示例（前 3 篇）：')
for m in mesh_articles['mesh'].head(3):
    print('  -', m.split('||'))


全部文章: 29907
有 MeSH : 17472 (58.4%)
被去掉  : 12435

主题词出现次数（含重复）: 182668
不同写法（未清洗）      : 44813

示例（前 3 篇）：
  - ['Animals', 'Appetite Depressants/*pharmacology', 'Eating/drug effects', 'Fasting', 'Glucagon-Like Peptide 2/administration & dosage/antagonists & inhibitors/*pharmacology', 'Male', 'Melanocyte-Stimulating Hormones/pharmacology', 'Microinjections', 'Peptide Fragments/pharmacology', 'Rats', 'Rats, Sprague-Dawley', 'Receptor, Melanocortin, Type 4/*drug effects', '*Solitary Nucleus']
  - ['Adult', 'Anatomic Landmarks', 'Asian People/ethnology', 'Body Weights and Measures', '*Cone-Beam Computed Tomography', 'Face/anatomy & histology/diagnostic imaging', 'Female', 'Humans', '*Imaging, Three-Dimensional', 'Lip/*anatomy & histology/*diagnostic imaging', 'Male', 'Regression Analysis', 'Reproducibility of Results', 'Young Adult']
  - ['*Biosensing Techniques']
